# BÁO CÁO: THU THẬP VÀ CHUẨN BỊ DỮ LIỆU ERA5 CHO MÔ HÌNH SPATIAL-TEMPORAL TRANSFORMER

---

## 1. Mục tiêu

Báo cáo này trình bày quy trình tải xuống và tổ chức bộ dữ liệu khí tượng từ nguồn **ERA5** của Copernicus. Mục tiêu chính là thu thập các biến số khí tượng quan trọng trên khu vực địa lý Việt Nam trong khoảng thời gian từ năm 2017 đến 2024. Bộ dữ liệu này sẽ được sử dụng làm đầu vào để huấn luyện một mô hình học sâu thuộc nhóm **Spatial-Temporal Transformer**, phục vụ cho các bài toán dự báo trong lĩnh vực khoa học khí hậu.

## 2. Thiết lập Môi trường và Xác thực

Bước đầu tiên là chuẩn bị môi trường làm việc trên Google Colab. Quá trình này bao gồm:

1.  **Cài đặt thư viện `cdsapi`**: Thư viện chính thức để tương tác với Copernicus Climate Data Store (CDS) API.
2.  **Cấu hình API Key**: Tạo file `.cdsapirc` chứa thông tin xác thực (URL và key) để `cdsapi` có thể tự động kết nối và gửi yêu cầu đến máy chủ.
3.  **Kết nối Google Drive**: Sử dụng Google Drive làm nơi lưu trữ lâu dài cho các file dữ liệu NetCDF (`.nc`) tải về. Điều này đảm bảo dữ liệu không bị mất sau khi phiên làm việc Colab kết thúc và dễ dàng truy cập lại sau này.

In [ ]:
!pip install cdsapi

In [ ]:
%%bash
cat > ~/.cdsapirc << EOL
url: https://cds.climate.copernicus.eu/api
key: 0f3c885c-49e2-4859-87cc-e57d5490d444
EOL

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Lựa chọn Tham số và Logic Tải Dữ liệu

Phần cốt lõi của quy trình là định nghĩa các tham số và xây dựng kịch bản tải dữ liệu. Các lựa chọn được đưa ra dựa trên yêu cầu của mô hình Spatial-Temporal và đặc thù của bài toán.

### 3.1. Các Lựa chọn chính

*   **Nguồn dữ liệu**: **ERA5 Reanalysis Single Levels**. Đây là bộ dữ liệu tái phân tích khí tượng toàn cầu với độ phân giải không gian và thời gian cao, cung cấp một cái nhìn nhất quán về trạng thái khí quyển.
*   **Định dạng file**: **NetCDF (`.nc`)**. Đây là định dạng tiêu chuẩn trong ngành khoa học khí hậu, tối ưu cho việc lưu trữ dữ liệu đa chiều (vĩ độ, kinh độ, thời gian) và dễ dàng xử lý bằng các thư viện như `xarray` hay `netCDF4`.
*   **Khu vực địa lý (Area)**: `[24, 102, 8, 110]` (North, West, South, East). Vùng bao này được chọn để phủ toàn bộ lãnh thổ Việt Nam và một phần Biển Đông lân cận.
*   **Các biến số (Variables)**: Một tập hợp gồm 10 biến số khí tượng bề mặt đã được lựa chọn. Đây là những yếu tố cơ bản, có ảnh hưởng trực tiếp đến các hiện tượng thời tiết và là những thuộc tính (features) quan trọng cho các mô hình dự báo, ví dụ: `2m_temperature` (nhiệt độ), `total_precipitation` (lượng mưa), `10m_u/v_component_of_wind` (thành phần gió), `surface_solar_radiation_downwards` (bức xạ mặt trời), v.v.
*   **Độ phân giải thời gian**: Dữ liệu được lấy theo từng giờ (`hourly`), nhưng để giảm kích thước và phù hợp với nhiều bài toán dự báo, tần suất được chọn là **2 giờ một lần** (`00:00`, `02:00`, ..., `22:00`).

### 3.2. Cấu trúc và Tối ưu hóa Quy trình Tải

Để quá trình tải dữ liệu lớn (8 năm) diễn ra ổn định và hiệu quả, kịch bản đã được thiết kế với các tính năng sau:

*   **Chia nhỏ theo giai đoạn (Chunking)**: Thay vì tải toàn bộ 8 năm trong một yêu cầu duy nhất (rất dễ thất bại), dữ liệu được chia thành 2 giai đoạn: `2017-2020` và `2021-2024`. Cách làm này giúp giảm tải cho API và dễ quản lý hơn.
*   **Tải riêng lẻ từng biến**: Vòng lặp bên trong sẽ tải tuần tự từng biến số. Mỗi biến trong một giai đoạn sẽ được lưu thành một file `.nc` riêng biệt (ví dụ: `era5_vn_2m_temperature_2017_2020.nc`). Điều này giúp cô lập lỗi và dễ dàng kiểm tra dữ liệu.
*   **Cơ chế Tiếp tục (Resume)**: Bằng cách sử dụng các biến `resume_label` và `resume_variable`, kịch bản có khả năng tự động bỏ qua các file đã được tải thành công trước đó nếu quá trình bị gián đoạn. Đây là một tính năng cực kỳ hữu ích, giúp tiết kiệm thời gian và tài nguyên.
*   **Xử lý lỗi và Tự động thử lại (Error Handling & Retry)**: Mỗi yêu cầu tải đều được đặt trong một khối `try...except`. Nếu có lỗi xảy ra (ví dụ: mất kết nối mạng, API quá tải), kịch bản sẽ tự động chờ 5 giây và thử lại thêm một lần nữa trước khi bỏ qua. Điều này tăng cường sự ổn định và khả năng phục hồi của quy trình.

In [ ]:
import os
import cdsapi
from datetime import datetime
import time

# ====== CÁC THAM SỐ CẤU HÌNH ======

# Thư mục lưu trữ dữ liệu trên Google Drive
base_dir = "/content/drive/MyDrive/Rain_Forecast_Fourier_Convolutional_Transformer/1_Data/Raw_Data"
os.makedirs(base_dir, exist_ok=True)

# Cấu hình điểm bắt đầu để tiếp tục nếu quá trình bị gián đoạn
resume_label = "2021_2024"               # Giai đoạn muốn tiếp tục (ví dụ: "2021_2024" hoặc None để bắt đầu từ đầu)
resume_variable = "total_precipitation"  # Biến đầu tiên muốn tải trong giai đoạn resume (hoặc None để chạy từ biến đầu tiên)
resume_found = False                     # Cờ trạng thái để kích hoạt cơ chế resume

# Kết nối tới CDS API client
c = cdsapi.Client()

# Danh sách các biến số cần tải
variables = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "mean_sea_level_pressure",
    "surface_pressure",
    "total_precipitation",
    "surface_solar_radiation_downwards",
    "skin_temperature",
    "total_column_water_vapour",
]

# Định nghĩa các tham số cho yêu cầu API
area = [24, 102, 8, 110]  # [North, West, South, East] - Bao phủ Việt Nam
times = [f"{h:02d}:00" for h in range(0, 24, 2)] # Dữ liệu 2 giờ một lần
days = [f"{d:02d}" for d in range(1, 32)]
months = [f"{m:02d}" for m in range(1, 13)]

# Chia quá trình tải thành các giai đoạn nhỏ hơn
periods = {
    "2017_2020": range(2017, 2021),
    "2021_2024": range(2021, 2025),
}

# ====== BẮT ĐẦU VÒNG LẶP TẢI DỮ LIỆU ======

for label, years in periods.items():
    # Kích hoạt cơ chế resume: bỏ qua các giai đoạn trước điểm resume
    if resume_label and label != resume_label and not resume_found:
        print(f"⏭️ Bỏ qua giai đoạn {label} (trước điểm resume)")
        continue
    resume_found = True

    print(f"\n======================")
    print(f"📅 BẮT ĐẦU GIAI ĐOẠN {label}")
    print(f"======================\n")

    for var in variables:
        # Kích hoạt cơ chế resume: bỏ qua các biến trước điểm resume
        if resume_variable and resume_found and var != resume_variable:
             print(f"⏭️ Bỏ qua {var} (trước điểm resume trong giai đoạn {label})")
             continue
        resume_variable = None # Vô hiệu hóa resume biến sau khi đã tìm thấy

        out_nc = os.path.join(base_dir, f"era5_vn_{var}_{label}.nc")

        # Bỏ qua nếu file đã tồn tại
        if os.path.exists(out_nc):
            print(f"⏩ Bỏ qua {var} ({label}) — đã tồn tại")
            continue

        # Xây dựng yêu cầu API cho biến và giai đoạn hiện tại
        req = {
            "product_type": "reanalysis",
            "format": "netcdf",
            "variable": [var],
            "year": [str(y) for y in years],
            "month": months,
            "day": days,
            "time": times,
            "area": area,
        }

        print(f"🚀 Tải {var} ({label})... ({datetime.now().strftime('%Y-%m-%d %H:%M:%S')})")

        # Vòng lặp thử lại (tối đa 2 lần)
        success = False
        for attempt in range(2):
            try:
                c.retrieve("reanalysis-era5-single-levels", req, out_nc)
                print(f"✅ Hoàn thành {out_nc}")
                success = True
                break # Thoát khỏi vòng lặp thử lại nếu thành công
            except Exception as e:
                print(f"❌ Lỗi khi tải {var} ({label}) (lần {attempt+1}): {e}")
                if attempt == 0:
                    print("🔁 Thử lại sau 5 giây...")
                    time.sleep(5)
                else:
                    print(f"⚠️ Bỏ qua {var} ({label}) sau 2 lần thất bại.")

print("\n🎯 Toàn bộ các giai đoạn đã xử lý xong.")

⏭️ Bỏ qua giai đoạn 2017_2020 (trước điểm resume)

📅 BẮT ĐẦU GIAI ĐOẠN 2021_2024

⏩ Bỏ qua 2m_temperature (2021_2024) — đã tồn tại
⏩ Bỏ qua 2m_dewpoint_temperature (2021_2024) — đã tồn tại
⏩ Bỏ qua 10m_u_component_of_wind (2021_2024) — đã tồn tại
⏩ Bỏ qua 10m_v_component_of_wind (2021_2024) — đã tồn tại
⏩ Bỏ qua mean_sea_level_pressure (2021_2024) — đã tồn tại
⏩ Bỏ qua surface_pressure (2021_2024) — đã tồn tại
🚀 Tải total_precipitation (2021_2024)... (2025-10-15 14:18:50.111956)


2025-10-15 14:18:50,911 INFO Request ID is 67db6efe-1a2a-4c46-8aa5-3e81f75602ef
INFO:ecmwf.datastores.legacy_client:Request ID is 67db6efe-1a2a-4c46-8aa5-3e81f75602ef
2025-10-15 14:18:51,063 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2025-10-15 14:19:12,778 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2025-10-15 14:19:24,311 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2025-10-15 14:19:41,553 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2025-10-15 14:41:16,973 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


bdb9ff4ba186d53e34dd3a2365e57994.nc:   0%|          | 0.00/45.2M [00:00<?, ?B/s]

✅ Hoàn thành /content/drive/MyDrive/era5_vn_min/era5_vn_total_precipitation_2021_2024.nc
🚀 Tải surface_solar_radiation_downwards (2021_2024)... (2025-10-15 14:41:20.764427)


2025-10-15 14:41:21,593 INFO Request ID is d55bafc5-1caa-40f2-a8f6-8cc5091d80da
INFO:ecmwf.datastores.legacy_client:Request ID is d55bafc5-1caa-40f2-a8f6-8cc5091d80da
2025-10-15 14:41:21,766 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2025-10-15 14:41:43,451 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2025-10-15 15:05:48,301 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

3ec24f0ed6da81a8347c232c47789bef.nc:   0%|          | 0.00/70.2M [00:00<?, ?B/s]

✅ Hoàn thành /content/drive/MyDrive/era5_vn_min/era5_vn_skin_temperature_2021_2024.nc
🚀 Tải total_column_water_vapour (2021_2024)... (2025-10-15 15:44:30.439021)


2025-10-15 15:44:31,214 INFO Request ID is cbc52fbf-a7d2-44d3-9f3e-04550b0f58c1
INFO:ecmwf.datastores.legacy_client:Request ID is cbc52fbf-a7d2-44d3-9f3e-04550b0f58c1
2025-10-15 15:44:31,436 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2025-10-15 15:47:24,688 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2025-10-15 16:14:59,706 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


e24716c2917d37ee47943c09abbfeb13.nc:   0%|          | 0.00/81.9M [00:00<?, ?B/s]

✅ Hoàn thành /content/drive/MyDrive/era5_vn_min/era5_vn_total_column_water_vapour_2021_2024.nc

🎯 Toàn bộ các giai đoạn đã xử lý xong.


## 4. Kiểm tra và Xác thực Dữ liệu

Sau khi quy trình tải hoàn tất, bước cuối cùng là kiểm tra lại thư mục đích. Đoạn mã dưới đây sẽ:

1.  Liệt kê tất cả các file có đuôi `.nc` trong thư mục `base_dir`.
2.  In ra đường dẫn đầy đủ và dung lượng (tính bằng Megabytes) của từng file.

Bước này giúp xác nhận rằng tất cả các file dự kiến đã được tải về thành công và cung cấp một cái nhìn tổng quan về khối lượng dữ liệu đã thu thập.

In [2]:
import os

folder = "/content/drive/MyDrive/Rain_Forecast_Fourier_Convolutional_Transformer/1_Data/Raw_Data" # Có thể thay bằng biến base_dir

# Lấy danh sách tất cả các file trong thư mục
files = os.listdir(folder)

# Lọc ra những file có đuôi .nc
nc_files = [f for f in files if f.endswith('.nc')]

print(f"--- KẾT QUẢ KIỂM TRA TRONG THƯ MỤC: {folder} ---")
# In đường dẫn đầy đủ và kích thước file
if not nc_files:
    print("Không tìm thấy file .nc nào.")
else:
    for f in sorted(nc_files): # Sắp xếp để dễ theo dõi
        full_path = os.path.join(folder, f)
        size_bytes = os.path.getsize(full_path)       # Kích thước file theo byte
        size_mb = size_bytes / (1024 * 1024)         # Chuyển đổi sang MB
        print(f"{full_path} — {size_mb:.2f} MB")

--- KẾT QUẢ KIỂM TRA TRONG THƯ MỤC: /content/drive/MyDrive/Rain_Forecast_Fourier_Convolutional_Transformer/1_Data/Raw_Data ---
/content/drive/MyDrive/Rain_Forecast_Fourier_Convolutional_Transformer/1_Data/Raw_Data/era5_vn_10m_u_component_of_wind_2017_2020.nc — 97.26 MB
/content/drive/MyDrive/Rain_Forecast_Fourier_Convolutional_Transformer/1_Data/Raw_Data/era5_vn_10m_u_component_of_wind_2021_2024.nc — 97.26 MB
/content/drive/MyDrive/Rain_Forecast_Fourier_Convolutional_Transformer/1_Data/Raw_Data/era5_vn_10m_v_component_of_wind_2017_2020.nc — 97.39 MB
/content/drive/MyDrive/Rain_Forecast_Fourier_Convolutional_Transformer/1_Data/Raw_Data/era5_vn_10m_v_component_of_wind_2021_2024.nc — 96.41 MB
/content/drive/MyDrive/Rain_Forecast_Fourier_Convolutional_Transformer/1_Data/Raw_Data/era5_vn_2m_dewpoint_temperature_2017_2020.nc — 72.02 MB
/content/drive/MyDrive/Rain_Forecast_Fourier_Convolutional_Transformer/1_Data/Raw_Data/era5_vn_2m_dewpoint_temperature_2021_2024.nc — 72.05 MB
/content/drive/

## 5. Hướng đi Tiếp theo

Sau khi hoàn tất việc thu thập dữ liệu, các bước tiếp theo trong dự án sẽ bao gồm:

1.  **Tiền xử lý (Preprocessing)**: Mở các file NetCDF, gộp dữ liệu theo chiều thời gian, xử lý các giá trị thiếu (nếu có), và chuẩn hóa dữ liệu.
2.  **Kỹ thuật thuộc tính (Feature Engineering)**: Tạo các cửa sổ trượt (sliding windows) theo không gian và thời gian để định dạng dữ liệu cho phù hợp với đầu vào của mô hình Transformer.
3.  **Huấn luyện mô hình**: Xây dựng và huấn luyện mô hình Spatial-Temporal Transformer với bộ dữ liệu đã chuẩn bị.
4.  **Đánh giá và Dự báo**: Đánh giá hiệu năng của mô hình trên tập kiểm tra và sử dụng nó để đưa ra các dự báo trong tương lai.